# Generating a general ledger with known anomalies

Most portfolio projects claim a workflow "detects anomalies". Almost none can say
**how many it caught and how many it missed**, because the ground truth is unknown.

This notebook builds a synthetic general ledger with anomalies deliberately planted
and recorded in an answer key. That makes detection measurable rather than asserted.

**Outputs**

| File | Contents |
|---|---|
| `general_ledger.csv` | ~168,000 line items across ~84,000 journal entries |
| `trial_balance.csv` | Account-level totals, for the completeness tie-out |
| `answer_key.csv` | Every planted anomaly, with its type and reason |

Financial year: 1 April 2025 to 31 March 2026 (Indian FY 2025–26).

**No client data is used anywhere.** Every figure is generated.

## Setup

The seed is fixed. That matters more than it looks: anyone who runs this notebook
gets a byte-identical ledger, which means the detection results in the reconnaissance
notebook are reproducible by a reviewer. Reproducibility is the difference between
a result and a claim.

In [4]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

SEED = 240726
rng = np.random.default_rng(SEED)

FY_START = date(2025, 4, 1)
FY_END   = date(2026, 3, 31)
N_DAYS   = (FY_END - FY_START).days + 1

APPROVAL_LIMIT = 500_000      # entries above this require approval
MATERIALITY    = 2_500_000    # planning materiality, drives the round-number test

print(f"financial year   {FY_START}  to  {FY_END}   ({N_DAYS} days)")
print(f"approval limit   {APPROVAL_LIMIT:,}")
print(f"materiality      {MATERIALITY:,}")

financial year   2025-04-01  to  2026-03-31   (365 days)
approval limit   500,000
materiality      2,500,000


## Chart of accounts

34 accounts across assets, liabilities, equity, income and expenses. The suspense
account (1900) matters — it is where several planted anomalies post, because in real
ledgers a suspense account is where awkward entries go to be forgotten.

In [5]:
COA = [
    ("1100","Cash in hand","Asset","D"),
    ("1110","Bank - current account","Asset","D"),
    ("1120","Bank - collection account","Asset","D"),
    ("1200","Trade receivables","Asset","D"),
    ("1210","Allowance for expected credit loss","Asset","C"),
    ("1300","Inventory - raw material","Asset","D"),
    ("1310","Inventory - finished goods","Asset","D"),
    ("1400","Prepaid expenses","Asset","D"),
    ("1410","Advance to suppliers","Asset","D"),
    ("1500","Property, plant and equipment","Asset","D"),
    ("1510","Accumulated depreciation","Asset","C"),
    ("1600","Input GST receivable","Asset","D"),
    ("1610","TDS receivable","Asset","D"),
    ("1700","Security deposits","Asset","D"),
    ("1900","Suspense account","Asset","D"),
    ("2100","Trade payables","Liability","C"),
    ("2110","Accrued expenses","Liability","C"),
    ("2120","Provision for employee benefits","Liability","C"),
    ("2200","Output GST payable","Liability","C"),
    ("2210","TDS payable","Liability","C"),
    ("2300","Short term borrowings","Liability","C"),
    ("2310","Interest accrued but not due","Liability","C"),
    ("2400","Unearned revenue","Liability","C"),
    ("3100","Share capital","Equity","C"),
    ("3200","Retained earnings","Equity","C"),
    ("4100","Revenue - product sales","Income","C"),
    ("4110","Revenue - service income","Income","C"),
    ("4200","Other income","Income","C"),
    ("4300","Interest income","Income","C"),
    ("5100","Cost of materials consumed","Expense","D"),
    ("5200","Employee benefit expense","Expense","D"),
    ("5210","Contract staff cost","Expense","D"),
    ("5300","Rent expense","Expense","D"),
    ("5310","Power and fuel","Expense","D"),
    ("5320","Repairs and maintenance","Expense","D"),
    ("5330","Travel and conveyance","Expense","D"),
    ("5340","Legal and professional fees","Expense","D"),
    ("5350","Advertisement and marketing","Expense","D"),
    ("5360","Insurance","Expense","D"),
    ("5370","Communication expense","Expense","D"),
    ("5380","Printing and stationery","Expense","D"),
    ("5400","Depreciation and amortisation","Expense","D"),
    ("5500","Finance cost","Expense","D"),
    ("5600","Bad debts written off","Expense","D"),
    ("5700","Miscellaneous expenses","Expense","D"),
]

coa = pd.DataFrame(COA, columns=["account_code","account_name","account_type","normal_side"])
ACC = {r.account_code: r for r in coa.itertuples()}
COST_CENTRES = ["CC-100","CC-200","CC-300","CC-400","CC-500","CC-600"]

print(f"{len(coa)} accounts")
coa.groupby("account_type").size().to_frame("count")

45 accounts


,count
account_type,
Asset,15
Equity,2
Expense,16
Income,4
Liability,8


## Users and the working calendar

Three user populations. The rare users exist so the "rare user ID" test has something
real to find — someone who posts four entries a year, one of them material, is exactly
the pattern an auditor wants surfaced.

The holiday calendar is indicative for India. Weekend and holiday postings become
the non-business-day test.

In [6]:
ROUTINE_USERS    = [f"U{str(i).zfill(3)}" for i in range(1, 13)]   # high volume
OCCASIONAL_USERS = [f"U{str(i).zfill(3)}" for i in range(13, 22)]  # medium volume
RARE_USERS       = ["U087", "U091", "U104"]                        # a few entries only
SYSTEM_USER      = "SYS-BATCH"
APPROVERS        = ["A001", "A002", "A003"]

HOLIDAYS = {
    date(2025,4,14), date(2025,4,18), date(2025,5,1),
    date(2025,8,15), date(2025,8,27), date(2025,10,2),
    date(2025,10,20), date(2025,10,21), date(2025,11,5),
    date(2025,12,25), date(2026,1,26), date(2026,3,4),
}

def business_day(d):
    return d.weekday() < 5 and d not in HOLIDAYS

ALL_DAYS          = [FY_START + timedelta(days=i) for i in range(N_DAYS)]
BUSINESS_DAYS     = [d for d in ALL_DAYS if business_day(d)]
NON_BUSINESS_DAYS = [d for d in ALL_DAYS if not business_day(d)]

print(f"business days      {len(BUSINESS_DAYS)}")
print(f"non-business days  {len(NON_BUSINESS_DAYS)}")

business days      249
non-business days  116


## Transaction templates

Each template is a realistic debit/credit pairing with a weight and an amount range.
The weights make some transaction types common and others rare, which is what gives
the "rare account pairing" test a believable baseline to work against.

In [7]:
TEMPLATES = [
    ("Sales invoice raised",     "1200","4100", 0.16, (15_000, 900_000)),
    ("Service invoice raised",   "1200","4110", 0.08, (25_000, 600_000)),
    ("Customer receipt",         "1110","1200", 0.14, (15_000, 900_000)),
    ("Purchase of raw material", "5100","2100", 0.11, (10_000, 700_000)),
    ("Supplier payment",         "2100","1110", 0.10, (10_000, 700_000)),
    ("Payroll for the month",    "5200","2120", 0.04, (400_000, 3_200_000)),
    ("Contract staff billing",   "5210","2100", 0.03, (40_000, 350_000)),
    ("Rent for the period",      "5300","2110", 0.02, (120_000, 450_000)),
    ("Electricity charges",      "5310","2110", 0.03, (30_000, 260_000)),
    ("Repairs and upkeep",       "5320","2100", 0.03, (8_000, 180_000)),
    ("Travel reimbursement",     "5330","1110", 0.04, (3_000, 90_000)),
    ("Professional fees billed", "5340","2100", 0.03, (25_000, 400_000)),
    ("Marketing spend",          "5350","2100", 0.03, (20_000, 500_000)),
    ("Insurance premium",        "5360","1400", 0.01, (40_000, 300_000)),
    ("Telephone and internet",   "5370","2110", 0.02, (5_000, 60_000)),
    ("Stationery purchase",      "5380","2100", 0.02, (2_000, 35_000)),
    ("Depreciation charge",      "5400","1510", 0.02, (150_000, 1_100_000)),
    ("Interest on borrowings",   "5500","2310", 0.02, (50_000, 700_000)),
    ("GST input booked",         "1600","2100", 0.03, (5_000, 120_000)),
    ("GST output booked",        "1200","2200", 0.03, (5_000, 150_000)),
    ("TDS deducted on payment",  "2100","2210", 0.02, (2_000, 70_000)),
    ("Advance to supplier",      "1410","1110", 0.02, (25_000, 400_000)),
    ("Bank interest credited",   "1110","4300", 0.01, (5_000, 90_000)),
]

tpl_narr = [t[0] for t in TEMPLATES]
tpl_dr   = [t[1] for t in TEMPLATES]
tpl_cr   = [t[2] for t in TEMPLATES]
tpl_w    = np.array([t[3] for t in TEMPLATES], dtype=float); tpl_w /= tpl_w.sum()
tpl_rng  = [t[4] for t in TEMPLATES]

print(f"{len(TEMPLATES)} transaction templates")

23 transaction templates


## Helper functions

Every entry is two lines — one debit, one credit — so the ledger balances by
construction. `flag()` records each planted anomaly in the answer key.

In [8]:
rows, answer_rows = [], []
entry_counter = 1_000_000

def next_entry_id():
    global entry_counter
    entry_counter += 1
    return f"JE{entry_counter}"

def add_entry(entry_id, post_dt, eff_dt, dr_acc, cr_acc, amount, user,
              entry_type, narration, approved, approver):
    """Append a balanced two-line journal entry."""
    common = dict(entry_id=entry_id, posting_date=post_dt, effective_date=eff_dt,
                  cost_centre=rng.choice(COST_CENTRES), user_id=user,
                  entry_type=entry_type, narration=narration,
                  approval_status=approved, approver_id=approver)
    rows.append({**common, "line_no":1, "account_code":dr_acc,
                 "account_name":ACC[dr_acc].account_name,
                 "account_type":ACC[dr_acc].account_type,
                 "debit":round(amount,2), "credit":0.0})
    rows.append({**common, "line_no":2, "account_code":cr_acc,
                 "account_name":ACC[cr_acc].account_name,
                 "account_type":ACC[cr_acc].account_type,
                 "debit":0.0, "credit":round(amount,2)})

def flag(entry_id, anomaly, detail):
    answer_rows.append(dict(entry_id=entry_id, anomaly_type=anomaly, detail=detail))

print("helpers ready")

helpers ready


## The routine population

84,000 ordinary entries. About 18% are manual, the rest system-generated — which
turns out to matter enormously for one of the tests, as the reconnaissance notebook
shows.

In [9]:
N_ROUTINE = 84_000

idx       = rng.choice(len(TEMPLATES), size=N_ROUTINE, p=tpl_w)
day_idx   = rng.integers(0, len(BUSINESS_DAYS), size=N_ROUTINE)
is_manual = rng.random(N_ROUTINE) < 0.18

for k in range(N_ROUTINE):
    t = idx[k]
    lo, hi = tpl_rng[t]
    amount = round(float(rng.uniform(lo, hi)), 2)
    post_dt = BUSINESS_DAYS[day_idx[k]]

    lag = int(rng.choice([0,0,0,0,1,1,2,3], size=1)[0])
    eff_dt = max(post_dt - timedelta(days=lag), FY_START)

    if is_manual[k]:
        user, etype = str(rng.choice(ROUTINE_USERS + OCCASIONAL_USERS)), "Manual"
    else:
        user, etype = SYSTEM_USER, "System"

    if amount > APPROVAL_LIMIT:
        approved, approver = "Approved", str(rng.choice(APPROVERS))
    else:
        approved, approver = "Not required", ""

    add_entry(next_entry_id(), post_dt, eff_dt, tpl_dr[t], tpl_cr[t],
              amount, user, etype, tpl_narr[t], approved, approver)

print(f"{N_ROUTINE:,} routine entries -> {len(rows):,} line items")
print(f"manual: {is_manual.sum():,}   system: {(~is_manual).sum():,}")

84,000 routine entries -> 168,000 line items
manual: 15,146   system: 68,854


## Planting the anomalies

Eight categories, each mapping to one test. These are the patterns that actually
surface problem entries in a real journal population — round-number postings,
out-of-hours activity, rare users, back-dating, amounts sitting just under an
approval threshold, unusual account pairings, telling narrations, and gaps in the
entry sequence.

In [10]:
# --- 1. Round-number manual postings above materiality -----------------
for _ in range(14):
    eid = next_entry_id()
    amt = float(rng.choice([2_500_000, 5_000_000, 7_500_000, 10_000_000]))
    d = BUSINESS_DAYS[int(rng.integers(0, len(BUSINESS_DAYS)))]
    add_entry(eid, d, d, "1900", "4200", amt, str(rng.choice(ROUTINE_USERS)),
              "Manual", "Balance transfer", "Approved", str(rng.choice(APPROVERS)))
    flag(eid, "round_number_above_materiality", f"Amount {amt:,.0f} exactly round")

# --- 2. Weekend and holiday postings -----------------------------------
for _ in range(22):
    eid = next_entry_id()
    d = NON_BUSINESS_DAYS[int(rng.integers(0, len(NON_BUSINESS_DAYS)))]
    amt = round(float(rng.uniform(300_000, 2_000_000)), 2)
    add_entry(eid, d, d, "5700", "2110", amt,
              str(rng.choice(ROUTINE_USERS + OCCASIONAL_USERS)), "Manual",
              "Month end provision", "Approved", str(rng.choice(APPROVERS)))
    flag(eid, "non_business_day_posting", f"Posted on {d.isoformat()} ({d.strftime('%A')})")

# --- 3. Rare user IDs posting material manual entries -------------------
for u in RARE_USERS:
    for _ in range(int(rng.integers(2, 5))):
        eid = next_entry_id()
        d = BUSINESS_DAYS[int(rng.integers(0, len(BUSINESS_DAYS)))]
        amt = round(float(rng.uniform(800_000, 4_000_000)), 2)
        add_entry(eid, d, d, "1900", "1110", amt, u, "Manual",
                  "Reclassification", "Approved", str(rng.choice(APPROVERS)))
        flag(eid, "rare_user", f"User {u} posts fewer than 10 entries in the year")

# --- 4. Back-dated entries near year end --------------------------------
for _ in range(18):
    eid = next_entry_id()
    eff  = date(2026,3,31) - timedelta(days=int(rng.integers(0,5)))
    post = date(2026,4,1)  + timedelta(days=int(rng.integers(8,40)))
    amt = round(float(rng.uniform(400_000, 3_000_000)), 2)
    add_entry(eid, post, eff, "5100", "2110", amt, str(rng.choice(ROUTINE_USERS)),
              "Manual", "Year end accrual", "Approved", str(rng.choice(APPROVERS)))
    flag(eid, "back_dated", f"Posted {(post-eff).days} days after effective date")

# --- 5. Amounts just below the approval threshold, one user -------------
splitter = "U007"
for _ in range(26):
    eid = next_entry_id()
    d = BUSINESS_DAYS[int(rng.integers(0, len(BUSINESS_DAYS)))]
    amt = round(float(rng.uniform(0.94, 0.999)) * APPROVAL_LIMIT, 2)
    add_entry(eid, d, d, "5340", "2100", amt, splitter, "Manual",
              "Consultancy charges", "Not required", "")
    flag(eid, "below_approval_threshold",
         f"{amt:,.0f} is {amt/APPROVAL_LIMIT:.1%} of the {APPROVAL_LIMIT:,} limit")

# --- 6. Rare account pairings -------------------------------------------
for dr_a, cr_a in [("4100","1900"), ("3200","1110"), ("1510","5700"), ("2400","4110")]:
    for _ in range(2):
        eid = next_entry_id()
        d = BUSINESS_DAYS[int(rng.integers(0, len(BUSINESS_DAYS)))]
        amt = round(float(rng.uniform(200_000, 1_800_000)), 2)
        add_entry(eid, d, d, dr_a, cr_a, amt, str(rng.choice(OCCASIONAL_USERS)),
                  "Manual", "Adjustment as discussed", "Approved", str(rng.choice(APPROVERS)))
        flag(eid, "rare_account_pairing", f"Dr {dr_a} / Cr {cr_a} occurs fewer than 5 times")

# --- 7. Narrations worth a second look ----------------------------------
for n in ["To plug difference in control account",
          "Reversal of earlier entry - to be corrected",
          "Temporary posting pending confirmation",
          "Adjustment to match management figures",
          "Squaring off old balance",
          "Correction entry as per instruction"]:
    for _ in range(3):
        eid = next_entry_id()
        d = BUSINESS_DAYS[int(rng.integers(0, len(BUSINESS_DAYS)))]
        amt = round(float(rng.uniform(150_000, 2_200_000)), 2)
        add_entry(eid, d, d, "1900", "2110", amt, str(rng.choice(ROUTINE_USERS)),
                  "Manual", n, "Approved", str(rng.choice(APPROVERS)))
        flag(eid, "narration_keyword", "Narration contains a review trigger word")

print(f"{len(answer_rows)} anomalies planted so far (sequence gaps added next)")

115 anomalies planted so far (sequence gaps added next)


## Assemble, and create sequence gaps

Nine complete entries are removed after assembly. That leaves real holes in the
entry ID sequence — which is what the Multi-Row Formula test hunts for in Alteryx.

In [11]:
gl = pd.DataFrame(rows).sort_values(["posting_date","entry_id","line_no"]).reset_index(drop=True)

all_ids  = gl["entry_id"].unique()
gap_ids  = rng.choice(all_ids, size=9, replace=False)
gl = gl[~gl["entry_id"].isin(gap_ids)].reset_index(drop=True)
for g in gap_ids:
    flag(g, "sequence_gap", "Entry ID absent from the ledger")

gl = gl[["entry_id","line_no","posting_date","effective_date","account_code",
         "account_name","account_type","cost_centre","user_id","entry_type",
         "narration","debit","credit","approval_status","approver_id"]]

print(f"line items   {len(gl):,}")
print(f"entries      {gl['entry_id'].nunique():,}")
gl.head()

line items   168,212
entries      84,106


,entry_id,line_no,posting_date,effective_date,account_code,account_name,account_type,cost_centre,user_id,entry_type,narration,debit,credit,approval_status,approver_id
0,JE1000312,1,2025-04-01,2025-04-01,5310,Power and fuel,Expense,CC-300,SYS-BATCH,System,Electricity charges,90337.93,0.00,Not required,
1,JE1000312,2,2025-04-01,2025-04-01,2110,Accrued expenses,Liability,CC-300,SYS-BATCH,System,Electricity charges,0.00,90337.93,Not required,
2,JE1000657,1,2025-04-01,2025-04-01,1110,Bank - current account,Asset,CC-500,SYS-BATCH,System,Customer receipt,180484.12,0.00,Not required,
3,JE1000657,2,2025-04-01,2025-04-01,1200,Trade receivables,Asset,CC-500,SYS-BATCH,System,Customer receipt,0.00,180484.12,Not required,
4,JE1002038,1,2025-04-01,2025-04-01,5310,Power and fuel,Expense,CC-600,SYS-BATCH,System,Electricity charges,73214.75,0.00,Not required,


## Trial balance and answer key

In [12]:
tb = (gl.groupby(["account_code","account_name","account_type"], as_index=False)
        .agg(total_debit=("debit","sum"), total_credit=("credit","sum")))
tb["net_balance"] = (tb["total_debit"] - tb["total_credit"]).round(2)
tb["total_debit"]  = tb["total_debit"].round(2)
tb["total_credit"] = tb["total_credit"].round(2)
tb = tb.sort_values("account_code").reset_index(drop=True)

answer = (pd.DataFrame(answer_rows).drop_duplicates()
            .sort_values(["anomaly_type","entry_id"]).reset_index(drop=True))

gl.to_csv("general_ledger.csv", index=False)
tb.to_csv("trial_balance.csv", index=False)
answer.to_csv("answer_key.csv", index=False)
gl.head(5000).to_csv("general_ledger_sample.csv", index=False)

print(f"general_ledger.csv   {len(gl):>9,} lines   {gl['entry_id'].nunique():>8,} entries")
print(f"trial_balance.csv    {len(tb):>9,} accounts")
print(f"answer_key.csv       {len(answer):>9,} planted anomalies")
print()
print(f"total debits    {gl['debit'].sum():>20,.2f}")
print(f"total credits   {gl['credit'].sum():>20,.2f}")
print(f"difference      {gl['debit'].sum()-gl['credit'].sum():>20,.2f}")
print()
print("planted anomalies by type")
print(answer["anomaly_type"].value_counts().to_string())

general_ledger.csv     168,212 lines     84,106 entries
trial_balance.csv           34 accounts
answer_key.csv             124 planted anomalies

total debits       30,523,085,595.19
total credits      30,523,085,595.19
difference                      0.00

planted anomalies by type
anomaly_type
below_approval_threshold          26
non_business_day_posting          22
back_dated                        18
narration_keyword                 18
round_number_above_materiality    14
rare_user                          9
sequence_gap                       9
rare_account_pairing               8


In [13]:
import os
os.getcwd()

'c:\\Users\\sadia\\Downloads\\Testing Journal Entries- python'

---

The ledger balances to zero, which it must — every entry was constructed as a
matched pair.

**Next:** `02_reconnaissance.ipynb` runs all eight tests in pandas and establishes
the target numbers the Alteryx workflow has to reproduce.

*Synthetic data generated for this project. No client data is used.*